# Comparative Results Summary

This notebook loads `experiments/full_evaluation.json` and creates a compact comparison table for all saved models.

It focuses on the most relevant metrics for the fraud detection project:
- PR-AUC
- F1
- Recall
- NLL
- Brier Score
- ECE


In [1]:
from pathlib import Path
import json
import pandas as pd

RESULTS_PATH = Path('experiments/full_evaluation.json')

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f'Could not find results file at: {RESULTS_PATH}\n'
        'Run the evaluation script first, or update RESULTS_PATH.'
    )

with RESULTS_PATH.open('r', encoding='utf-8') as f:
    results = json.load(f)

print('Loaded:', RESULTS_PATH)


Loaded: experiments\full_evaluation.json


In [2]:
rows = []

for model_name, model_info in results['models'].items():
    for split_name in ['validation', 'test']:
        split_info = model_info[split_name]
        cls = split_info['classification_metrics']
        prob = split_info['proper_scoring_rules']
        cal = split_info['calibration_metrics']

        rows.append({
            'model': model_name,
            'split': split_name,
            'pr_auc': cls['pr_auc'],
            'f1': cls['f1'],
            'recall': cls['recall'],
            'precision': cls['precision'],
            'roc_auc': cls['roc_auc'],
            'nll': prob['nll'],
            'brier_score': prob['brier_score'],
            'ece': cal['ece'],
            'mce': cal['mce'],
        })

df = pd.DataFrame(rows)
df.head()


,model,split,pr_auc,f1,recall,precision,roc_auc,nll,brier_score,ece,mce
0,baseline_results/logistic_regression,validation,0.680711,0.767568,0.717172,0.825581,0.974728,0.101567,0.022154,0.064351,0.838180
1,baseline_results/logistic_regression,test,0.717874,0.808081,0.816327,0.800000,0.972552,0.104314,0.022053,0.064444,0.850722
2,baseline_results/random_forest,validation,0.811919,0.824176,0.757576,0.903614,0.946509,0.007689,0.000539,0.000301,0.447143
3,baseline_results/random_forest,test,0.851120,0.868132,0.806122,0.940476,0.956872,0.006169,0.000434,0.000293,0.512917
4,boosting_results/xgboost,validation,0.825370,0.850829,0.777778,0.939024,0.975353,0.003416,0.000549,0.000338,0.838214


In [3]:
# Compact table focused on the test split
test_df = df[df['split'] == 'test'].copy()
test_df = test_df.sort_values(by=['pr_auc', 'f1', 'nll'], ascending=[False, False, True])

compact_cols = [
    'model', 'pr_auc', 'f1', 'recall', 'precision', 'nll', 'brier_score', 'ece'
]

compact_table = test_df[compact_cols].reset_index(drop=True)
compact_table.style.format({
    'pr_auc': '{:.4f}',
    'f1': '{:.4f}',
    'recall': '{:.4f}',
    'precision': '{:.4f}',
    'nll': '{:.4f}',
    'brier_score': '{:.4f}',
    'ece': '{:.4f}',
})


,model,pr_auc,f1,recall,precision,nll,brier_score,ece
0,boosting_results/xgboost,0.8697,0.8525,0.7959,0.9176,0.0031,0.0005,0.0006
1,baseline_results/random_forest,0.8511,0.8681,0.8061,0.9405,0.0062,0.0004,0.0003
2,bnn_results/bayesian_neural_network,0.7262,0.7706,0.8571,0.7000,0.1066,0.0235,0.0832
3,baseline_results/logistic_regression,0.7179,0.8081,0.8163,0.8000,0.1043,0.0221,0.0644


In [4]:
# Validation vs test side by side
pivot_table = df.pivot(index='model', columns='split', values=['pr_auc', 'f1', 'recall', 'nll', 'brier_score', 'ece'])
pivot_table = pivot_table.sort_values(by=('pr_auc', 'test'), ascending=False)
pivot_table.style.format('{:.4f}')


In [5]:
# Optional: export compact table to CSV
output_csv = Path('experiments/model_comparison_table.csv')
compact_table.to_csv(output_csv, index=False)
print(f'Saved compact comparison table to: {output_csv}')


Saved compact comparison table to: experiments\model_comparison_table.csv
